# Training model - StrtifiedKFold version

In [21]:
## import libraries
import pandas as pd
import numpy as np 
import imblearn
from matplotlib.pyplot import figure
from sklearn.utils import shuffle
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from imblearn.under_sampling import RandomUnderSampler
from sklearn import metrics
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from collections import Counter
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [9]:
df = pd.read_csv("../data/card_transdata.csv")

In [ ]:
rs = 123
X = df.loc[ : , df.columns != 'fraud']
y = df['fraud'].astype('int')

In [26]:
def skf(model):
    skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=rs
    )
    pipeline = Pipeline([
        ("scaler", QuantileTransformer()),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=skf,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"])
    for metrics in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
     print(metrics, scores[f"test_{metrics}"].mean())

### Model KNeighbors

In [27]:

model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

skf(model=model)


accuracy 0.994215
precision 0.9700989501830719
recall 0.9635138614699947
f1 0.9667922843606904
roc_auc 0.9988600657116834


## Model Random Forest

In [28]:
rf = RandomForestClassifier(n_estimators=100, random_state=rs, n_jobs=-1)
skf(rf)

accuracy 0.999992
precision 1.0
recall 0.9999084707463363
f1 0.9999542314458291
roc_auc 0.999999999373139


### Model XGBOOST

In [32]:
import xgboost as xgb

In [33]:
ratio_balance = (y == 0).sum() / (y == 1).sum()
xg_model = xgb.XGBClassifier(scale_pos_weight=ratio_balance, random_state=rs, n_jobs=-1)
skf(xg_model)

accuracy 0.997944
precision 0.9781634994171166
recall 0.9987757816691989
f1 0.9883615673042723
roc_auc 0.999970783677106


### Model Light GBM

In [37]:
import lightgbm as lgb
l_gbm = lgb.LGBMClassifier(is_unbalanced=True,random_state=rs, n_jobs=-1)


In [38]:
skf(l_gbm)

[LightGBM] [Warning] Unknown parameter: is_unbalanced
[LightGBM] [Warning] Unknown parameter: is_unbalanced
[LightGBM] [Info] Number of positive: 69923, number of negative: 730077
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015355 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 773
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087404 -> initscore=-2.345755
[LightGBM] [Info] Start training from score -2.345755
[LightGBM] [Warning] Unknown parameter: is_unbalanced
[LightGBM] [Warning] Unknown parameter: is_unbalanced
[LightGBM] [Warning] Unknown parameter: is_unbalanced
[LightGBM] [Warning] Unknown parameter: is_unbalanced
[LightGBM] [Info] Number of positive: 69923, number of negative: 730077
[LightGBM] [Info] Auto-choosing row-wise multi

### Conclusion

After comparing KNN, Random Forest, XGBoost, and LightGBM using 5-fold Stratified K-Fold cross-validation, **Random Forest achieved the best overall performance**, with nearly perfect scores across all metrics. However, due to its exceptionally high performance, further validation is needed to rule out data leakage before considering it the final model.
